In [ ]:
import json
import tiktoken

def extract_think(text: str) -> str:
    if "think" in text:
        return text.split("<think>")[-1].split("</think>")[0].strip()
    elif "<think>" in text:
        return text.split("<think>")[-1].split("</think>")[0].strip()
    else:
        return text.split("<think>")[-1].split("</think>")[0].strip()

def count_tokens(text: str) -> int:
    encoding = tiktoken.get_encoding("cl100k_base")
    tokens = encoding.encode(text)
    return len(tokens)

In [5]:
models = [
    "DeepSeek-V3",
    "DeepSeek-R1",
    "Qwen2.5-72B-Ins",
    "QwQ-32B",
]



In [6]:
for model in models:
    # print("==" * 10 + f" {model} " + "==" * 10)
    filepath1 = f"results/codegeneration/{model}/lcb_generation.json"
    filepath2 = f"results/codeexecution/{model}/lcb_output.json"
    filepath3 = f"results/codeexecution/{model}/crux_output_record.json"
    
    output_text = f"{model} & "
    for filepath in [filepath1, filepath2, filepath3]:
        
        if "lcb" in filepath:
            with open(filepath.replace(".json", "_eval.json"), "r") as f:
                data = json.load(f)
                score = data[0]["pass@1"] if "output" in filepath else data[0]["pass@1"] * 100
        else:
            with open(filepath.replace("_record", "_scored"), "r") as f:
                data = json.load(f)
                score = data["pass_at_1"]
        
        with open(filepath, "r") as f:
            data = json.load(f)
        total_tokens = 0
        max_tokens = 0
        for entry in data:
            # print(entry)
            token_count = count_tokens(extract_think(entry["output_list"][0]))
            total_tokens += token_count
            max_tokens = max(max_tokens, token_count)
        avg_tokens = total_tokens / len(data)
        output_text += f"${score:.2f}$ & ${avg_tokens:.0f}$ & ${max_tokens:.0f}$ & "
    output_text = output_text[:-2] + "\\\\"
    print(output_text)


DeepSeek-V3 & $76.32$ & $722$ & $1690$ & $92.07$ & $428$ & $1938$ & $88.00$ & $272$ & $1856$ \\
DeepSeek-R1 & $81.60$ & $3169$ & $25887$ & $98.96$ & $895$ & $4987$ & $92.62$ & $903$ & $5012$ \\
Qwen2.5-72B-Ins & $44.42$ & $240$ & $680$ & $90.40$ & $491$ & $1942$ & $78.25$ & $295$ & $995$ \\
QwQ-32B & $86.30$ & $6283$ & $32693$ & $98.96$ & $1518$ & $9206$ & $93.25$ & $1391$ & $13718$ \\
